In [3]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl


# ============================================================
# SETTINGS
# ============================================================

DATA_PATH = Path(
    r"C:\Users\abrosnan\Documents\PhD_1\Experiments\Dominance hierachy\Alpha LSD\LSD 2\LSD_POK_3\LSD_DOMINANT\codex_2026-06-21\results\chasings_df.parquet"
)

# Set to None to display complete animal IDs
SHORT_ID_LENGTH = 4

OUTPUT_PATH = DATA_PATH.parent / "initiated_vs_received_scatter.html"


# ============================================================
# LOAD AND CHECK DATA
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"\nCould not find the file:\n{DATA_PATH}\n\n"
        "Check the path pasted into DATA_PATH."
    )

df = pl.read_parquet(DATA_PATH)

required_columns = {"chaser", "chased", "chasings"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}\n"
        f"Available columns: {df.columns}"
    )


# ============================================================
# PREPARE ANIMAL IDS
# ============================================================

df = df.with_columns(
    pl.col("chaser").cast(pl.String),
    pl.col("chased").cast(pl.String),
)

if SHORT_ID_LENGTH is not None:
    df = df.with_columns(
        pl.col("chaser").str.slice(0, SHORT_ID_LENGTH),
        pl.col("chased").str.slice(0, SHORT_ID_LENGTH),
    )


# ============================================================
# CALCULATE INITIATED AND RECEIVED CHASES
# ============================================================

initiated = (
    df.group_by("chaser")
    .agg(pl.col("chasings").sum().alias("initiated"))
    .rename({"chaser": "animal"})
)

received = (
    df.group_by("chased")
    .agg(pl.col("chasings").sum().alias("received"))
    .rename({"chased": "animal"})
)

summary = (
    initiated.join(
        received,
        on="animal",
        how="full",
        coalesce=True,
    )
    .with_columns(
        pl.col("initiated").fill_null(0),
        pl.col("received").fill_null(0),
    )
    # Cast to signed integers before subtraction to prevent
    # unsigned-integer underflow when received > initiated.
    .with_columns(
        (
            pl.col("initiated").cast(pl.Int64)
            - pl.col("received").cast(pl.Int64)
        ).alias("net_chasing")
    )
    .sort("initiated", descending=True)
)


# ============================================================
# CREATE SCATTERPLOT
# ============================================================

scatter_fig = go.Figure()

scatter_fig.add_scatter(
    x=summary["initiated"].to_list(),
    y=summary["received"].to_list(),
    mode="markers+text",
    text=summary["animal"].to_list(),
    textposition="top center",
    marker={"size": 12},
    customdata=summary["net_chasing"].to_list(),
    hovertemplate=(
        "Animal: %{text}<br>"
        "Initiated: %{x:,}<br>"
        "Received: %{y:,}<br>"
        "Net chasing: %{customdata:,}"
        "<extra></extra>"
    ),
)

maximum = max(
    summary["initiated"].max(),
    summary["received"].max(),
)

scatter_fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=maximum,
    y1=maximum,
    line={
        "dash": "dash",
        "color": "grey",
    },
)

scatter_fig.update_layout(
    title="Individual chasing profiles",
    xaxis_title="Chasing events initiated",
    yaxis_title="Chasing events received",
    template="plotly_white",
    height=650,
    width=850,
)

scatter_fig.update_xaxes(range=[0, maximum * 1.05])
scatter_fig.update_yaxes(
    range=[0, maximum * 1.05],
    scaleanchor="x",
    scaleratio=1,
)

scatter_fig.show()


# ============================================================
# SAVE SCATTERPLOT
# ============================================================

scatter_fig.write_html(OUTPUT_PATH)

print(f"Scatterplot saved to:\n{OUTPUT_PATH}")

Scatterplot saved to:
C:\Users\abrosnan\Documents\PhD_1\Experiments\Dominance hierachy\Alpha LSD\LSD 2\LSD_POK_3\LSD_DOMINANT\codex_2026-06-21\results\initiated_vs_received_scatter.html
